# RetinaGuard: Lesion-Guided, Causally Informed Diabetic Retinopathy Diagnosis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harsha-Vardan/RetinaGuard/blob/main/notebooks/RetinaGuard_Colab_Training.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-RetinaGuard-blue.svg)](https://github.com/Harsha-Vardan/RetinaGuard)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0%2B-ee4c2c.svg)](https://pytorch.org)

---

## Overview
This notebook provides the **1-click GPU training and evaluation pipeline** for the **RetinaGuard** framework on Google Colab (compatible with free T4 or premium V100/A100 GPUs).

### Key Innovations Implemented:
1. **Retinal Preprocessing**: Circular FOV detection, aspect-ratio padding to 512x512, and LAB CLAHE contrast normalization.
2. **Spatial Lesion Attention**: Supervised auxiliary segmentation branch detecting microaneurysms, exudates, and hemorrhages ($\\mathcal{L}_{lesion}$).
3. **Causal Representation Disentanglement**: Separates disease features ($Z_{dis}$) from hospital/camera domain shortcuts ($Z_{dom}$) via **Gradient Reversal Layer (GRL)** and **Covariance Independence Penalty** ($Z_{dis} \\perp Z_{dom}$).
4. **Clinical Attribution & Explainability**: Learned spatial attention maps and Grad-CAM overlays evaluated with Attribution IoU and Dice against clinical ground truths.


### Step 1: Environment & GPU Setup
Verify GPU acceleration and clone the repository if running on Colab.

In [ ]:
import os
import sys
from pathlib import Path
import torch

print("=== Hardware Acceleration Inspection ===")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("Running on CPU (Tip: For 10x faster training, select Runtime -> Change runtime type -> T4 GPU)")

# If running inside Google Colab, clone the repository
if 'google.colab' in str(get_ipython()):
    print("\nConfiguring Google Colab environment...")
    if not Path("RetinaGuard").exists():
        !git clone https://github.com/Harsha-Vardan/RetinaGuard.git
    %cd RetinaGuard
    !pip install -q -r requirements.txt
    print("Dependencies installed and workspace configured!")
else:
    print("\nRunning in local/offline environment.")

### Step 2: Verify Test Suite & Data Manifests
Run unit tests across models, metrics, preprocessing, and attribution modules to guarantee integrity.

In [ ]:
!pytest tests -q --tb=short

### Step 3: Train the Unified RetinaGuard Multi-Task Framework
Execute end-to-end multi-task training with PyTorch Mixed Precision (`torch.cuda.amp.autocast`).

$$\\mathcal{L}_{total} = \\mathcal{L}_{CE}(y, \\hat{y}) + \\lambda_1 \\mathcal{L}_{BCE}(M, \\hat{M}) + \\lambda_2 \\mathcal{L}_{dom}(d, \\hat{d}) + \\lambda_3 \\mathcal{L}_{cov}(Z_{dis}, Z_{dom})$$

In [ ]:
# Execute RetinaGuard multi-task training
# Adjust epochs as needed (e.g. --epochs 30 for full convergence, or --smoke-test for a quick dry-run)
!python -m src.training.train_retinaguard --epochs 10 --batch-size 16 --architecture resnet18

### Step 4: Cross-Domain Generalization Benchmarking
Evaluate standard Empirical Risk Minimization (ERM) vs. RetinaGuard under hospital domain shift.

In [ ]:
# Run head-to-head comparative benchmark
!python -m src.evaluation.benchmark_comparison

### Step 5: Visual Explainability & Clinical Attribution
Load the trained model and visualize the learned spatial lesion attention maps alongside clinical fundus photography.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Find latest generated explanation figure or sample
fig_paths = list(Path("outputs/figures").glob("*comparison*.png")) + list(Path("outputs/figures").glob("*explanation*.png"))

if fig_paths:
    sample_fig = fig_paths[0]
    print(f"Displaying clinical explanation figure: {sample_fig}")
    img = Image.open(sample_fig)
    plt.figure(figsize=(14, 7), dpi=150)
    plt.imshow(img)
    plt.axis('off')
    plt.title("RetinaGuard: Multi-Panel Clinical Attribution & Heatmap Overlay", fontsize=14, pad=12)
    plt.show()
else:
    print("Run src/inference/predict.py or benchmark_comparison to generate figures.")

### Step 6: Single-Image Clinical Prediction CLI
Test any custom patient fundus photograph and generate a comprehensive diagnostic report.

In [ ]:
# Run unified inference with automatic architecture detection and heatmap generation
!python -m src.inference.predict --image "data/raw/IDRiD/B. Disease Grading/1. Original Images/a. Training Set/IDRiD_001.jpg" --save-heatmap outputs/figures/colab_sample_explanation.png